# 🏥 Sports Injury Prediction using Machine Learning

An end-to-end Machine Learning project for predicting sports injury risk using multiple classification models.

In [ ]:
# ============================================================
# SPORTS INJURY PREDICTION USING MACHINE LEARNING
# ============================================================

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    RocCurveDisplay
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

print("=" * 60)
print("SPORTS INJURY PREDICTION USING MACHINE LEARNING")
print("=" * 60)

# ============================================================
# CREATE PROJECT DIRECTORIES
# ============================================================

os.makedirs("models", exist_ok=True)
os.makedirs("preprocessing", exist_ok=True)

# ============================================================
# GENERATE SYNTHETIC DATASET
# ============================================================

def generate_sports_injury_dataset(n_samples=1000, random_state=42):
    np.random.seed(random_state)

    age = np.random.randint(18, 40, n_samples)
    training_load = np.random.uniform(1, 10, n_samples)
    previous_injuries = np.random.poisson(1.5, n_samples)
    sleep_quality = np.random.uniform(1, 10, n_samples)
    nutrition_score = np.random.uniform(1, 10, n_samples)
    muscle_fatigue = np.random.uniform(1, 10, n_samples)
    joint_flexibility = np.random.uniform(1, 10, n_samples)
    hydration_level = np.random.uniform(1, 10, n_samples)

    playing_surface = np.random.choice(
        ["Grass", "Hard", "Synthetic"], n_samples
    )

    risk_score = (
        0.30 * training_load
        + 0.70 * previous_injuries
        + 0.40 * muscle_fatigue
        - 0.30 * sleep_quality
        - 0.20 * nutrition_score
        - 0.25 * hydration_level
        - 0.20 * joint_flexibility
        + np.random.normal(0, 2, n_samples)
    )

    threshold = np.median(risk_score)
    injury_risk = (risk_score > threshold).astype(int)

    return pd.DataFrame({
        "Age": age,
        "Training_Load": training_load,
        "Previous_Injuries": previous_injuries,
        "Sleep_Quality": sleep_quality,
        "Nutrition_Score": nutrition_score,
        "Muscle_Fatigue": muscle_fatigue,
        "Joint_Flexibility": joint_flexibility,
        "Hydration_Level": hydration_level,
        "Playing_Surface": playing_surface,
        "Injury_Risk": injury_risk
    })


# ============================================================
# DATASET CREATION
# ============================================================

df = generate_sports_injury_dataset(
    n_samples=1000,
    random_state=RANDOM_STATE
)

print("\\nDataset Shape:", df.shape)
print("\\nFirst 5 Rows:")
print(df.head())


# ============================================================
# EXPLORATORY DATA ANALYSIS
# ============================================================

print("\\n" + "=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

print("\\nDataset Information:")
df.info()

print("\\nMissing Values:")
print(df.isnull().sum())

print("\\nStatistical Summary:")
print(df.describe())


# ============================================================
# INJURY RISK DISTRIBUTION
# ============================================================

plt.figure(figsize=(7, 5))

sns.countplot(data=df, x="Injury_Risk")

plt.title("Injury Risk Distribution")
plt.xlabel("Injury Risk")
plt.ylabel("Number of Athletes")
plt.xticks([0, 1], ["Low Risk", "High Risk"])

plt.show()


# ============================================================
# CORRELATION HEATMAP
# ============================================================

plt.figure(figsize=(10, 7))

correlation_matrix = df.select_dtypes(include=np.number).corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Feature Correlation Heatmap")
plt.show()


# ============================================================
# DATA PREPROCESSING
# ============================================================

df_encoded = pd.get_dummies(
    df,
    columns=["Playing_Surface"],
    drop_first=True
)

X = df_encoded.drop("Injury_Risk", axis=1)
y = df_encoded["Injury_Risk"]

print("\\nFeature Shape:", X.shape)
print("\\nFeature Names:")
print(list(X.columns))

joblib.dump(
    list(X.columns),
    "preprocessing/feature_names.pkl"
)


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)


# ============================================================
# DUMMY BASELINE MODEL
# ============================================================

dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train, y_train)

dummy_predictions = dummy_model.predict(X_test)

dummy_accuracy = accuracy_score(
    y_test,
    dummy_predictions
)

print(
    f"\\nDummy Baseline Accuracy: {dummy_accuracy:.4f}"
)


# ============================================================
# DEFINE MACHINE LEARNING MODELS
# ============================================================

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE
            )
        )
    ]),

    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        (
            "model",
            SVC(
                probability=True,
                random_state=RANDOM_STATE
            )
        )
    ]),

    "Random Forest": Pipeline([
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE
            )
        )
    ])
}


# ============================================================
# 5-FOLD CROSS VALIDATION
# ============================================================

print("\\n" + "=" * 60)
print("5-FOLD CROSS VALIDATION")
print("=" * 60)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1 Score": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_results_df = pd.DataFrame(cv_results)

print("\\nCross Validation Results:")
print(
    cv_results_df.sort_values(
        by="F1 Score",
        ascending=False
    )
)


# ============================================================
# TRAIN AND EVALUATE MODELS
# ============================================================

print("\\n" + "=" * 60)
print("MODEL TRAINING AND EVALUATION")
print("=" * 60)

results = []
trained_models = {}

for name, model in models.items():

    print("\\n" + "-" * 50)
    print(name)
    print("-" * 50)

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, probabilities)

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

    trained_models[name] = model

results_df = pd.DataFrame(results)

print("\\nFinal Model Results:")
print(
    results_df.sort_values(
        by="F1 Score",
        ascending=False
    )
)


# ============================================================
# MODEL COMPARISON
# ============================================================

plt.figure(figsize=(10, 6))

sns.barplot(
    data=results_df,
    x="Model",
    y="F1 Score"
)

plt.title("Model Comparison Based on F1 Score")
plt.xlabel("Machine Learning Model")
plt.ylabel("F1 Score")
plt.ylim(0, 1)
plt.xticks(rotation=15)

plt.show()


# ============================================================
# CONFUSION MATRICES
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

for ax, (name, model) in zip(
    axes,
    trained_models.items()
):

    predictions = model.predict(X_test)

    cm = confusion_matrix(
        y_test,
        predictions
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax
    )

    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


# ============================================================
# ROC CURVE COMPARISON
# ============================================================

plt.figure(figsize=(8, 6))

for name, model in trained_models.items():

    probabilities = model.predict_proba(X_test)[:, 1]

    RocCurveDisplay.from_predictions(
        y_test,
        probabilities,
        name=name
    )

plt.title("ROC Curve Comparison")
plt.show()


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

print("\\n" + "=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

rf_pipeline = trained_models["Random Forest"]

random_forest = rf_pipeline.named_steps["model"]

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": random_forest.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)


# ============================================================
# FEATURE IMPORTANCE VISUALIZATION
# ============================================================

plt.figure(figsize=(10, 6))

sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)

plt.title(
    "Feature Importance for Injury Risk Prediction"
)

plt.show()


# ============================================================
# SELECT BEST MODEL
# ============================================================

best_model_name = results_df.loc[
    results_df["F1 Score"].idxmax(),
    "Model"
]

best_model = trained_models[
    best_model_name
]

best_model_f1 = results_df.loc[
    results_df["F1 Score"].idxmax(),
    "F1 Score"
]

print("\\n" + "=" * 60)
print(f"BEST MODEL: {best_model_name}")
print(f"BEST F1 SCORE: {best_model_f1:.4f}")
print("=" * 60)


# ============================================================
# SAVE BEST MODEL
# ============================================================

joblib.dump(
    best_model,
    "models/injury_risk_pipeline.pkl"
)

print("\\nBest model saved successfully!")
print("Location: models/injury_risk_pipeline.pkl")


# ============================================================
# FINAL PROJECT SUMMARY
# ============================================================

best_cv_model = cv_results_df.loc[
    cv_results_df["F1 Score"].idxmax(),
    "Model"
]

best_cv_f1 = cv_results_df.loc[
    cv_results_df["F1 Score"].idxmax(),
    "F1 Score"
]

print("\\n" + "=" * 65)
print("SPORTS INJURY PREDICTION - FINAL SUMMARY")
print("=" * 65)

print(f"\\nDataset Size: {df.shape}")
print(f"\\nDummy Baseline Accuracy: {dummy_accuracy:.4f}")
print(f"\\nBest Test Model: {best_model_name}")
print(f"Best Test F1 Score: {best_model_f1:.4f}")
print(f"\\nBest Cross Validation Model: {best_cv_model}")
print(f"Best Cross Validation F1 Score: {best_cv_f1:.4f}")

print("\\nSaved Model:")
print("models/injury_risk_pipeline.pkl")

print("\\nSaved Features:")
print("preprocessing/feature_names.pkl")

print("\\n" + "=" * 65)
print("PROJECT EXECUTED SUCCESSFULLY!")
print("=" * 65)
